# Parsing the IEEE 1599 specimen

In [1]:
from pprint import pprint
from typing import Optional

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

import processing.tta.parsing
from processing import utils

def schema_to_dict(schema) -> dict:
    """
    Recursively converts a PyArrow schema or field into a dictionary structure
    suitable for pretty printing.

    Args:
        schema: A pyarrow.Schema or pyarrow.Field object.

    Returns:
        A dictionary representing the schema structure.
    """
    if isinstance(schema, (pa.Schema, pa.StructType)):
        return {field.name: schema_to_dict(field.type) for field in schema}
    if isinstance(schema, pa.ListType):
        return [schema_to_dict(schema.value_type)]
    if isinstance(schema, pa.Field):
        return schema_to_dict(schema.type)
    # For primitive types, return their string representation
    return str(schema)

def pretty_print_table_schema(table: pa.Schema):
    pprint(schema_to_dict(table.schema))

C:\Users\dmuri\AppData\Roaming\Python\Python312\site-packages\partitura\__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Loading XML as JSON structure

In [2]:
ieee = utils.xml2json("../ieee1599/gymnopedie_01.xml")
ieee

{'ieee1599': {'@creator': 'Finale Plugin',
  '@version': '1.0',
  'general': {'description': {'main_title': 'Gymnopédie No. 1',
    'author': {'@type': 'composer', '#text': 'Erik Satie'},
    'work_title': '3 Gymnopédies'}},
  'logic': {'spine': {'event': [{'@hpos': '0',
      '@id': 'Clef_part_1_1',
      '@timing': '0'},
     {'@hpos': '0', '@id': 'Clef_part_2_1', '@timing': '0'},
     {'@hpos': '0', '@id': 'KeySignature_part_1_1', '@timing': '0'},
     {'@hpos': '0', '@id': 'KeySignature_part_2_1', '@timing': '0'},
     {'@hpos': '0', '@id': 'TimeSignature_part_1_1', '@timing': '0'},
     {'@hpos': '0', '@id': 'TimeSignature_part_2_1', '@timing': '0'},
     {'@hpos': '0', '@id': 'part_1_voice0_measure1_ev0', '@timing': '0'},
     {'@hpos': '0', '@id': 'part_2_voice0_measure1_ev0', '@timing': '0'},
     {'@hpos': '0', '@id': 'part_2_voice1_measure1_ev0', '@timing': '0'},
     {'@hpos': '1024', '@id': 'part_2_voice1_measure1_ev1', '@timing': '1024'},
     {'@hpos': '2048', '@id': 'par

In [3]:
ieee.keys()

dict_keys(['ieee1599'])

In [4]:
dfs = processing.tta.parsing.partially_explode_json(
    ieee,
    explosion_paths=[
        ["measure", "measure.voice", ["measure.voice.rest", "measure.voice.chord"]],
        ["graphic_instance", "graphic_instance.graphic_event"],
        ["track_indexing.track_event"],
        ["segmentation.segment", "segmentation.segment.segment_event"],
        ["petri_net", "petri_net.place"]
    ],
)
dfs.keys()

dict_keys(['ieee1599.general.description.author', 'ieee1599.logic.spine.event', 'ieee1599.logic.los.staff_list.staff', 'ieee1599.logic.los.part', 'ieee1599.structural.analysis', 'ieee1599.structural.petri_nets', 'ieee1599.notational.graphic_instance_group', 'ieee1599.audio.track'])

## Parsing spine

In [5]:
spine = dfs["ieee1599.logic.spine.event"]
spine.show_meta()
spine

{'ieee1599.@creator': 'Finale Plugin',
 'ieee1599.@version': '1.0',
 'xml_node': 'ieee1599.logic.spine.event'}


,@hpos,@id,@timing
0,0,Clef_part_1_1,0
1,0,Clef_part_2_1,0
2,0,KeySignature_part_1_1,0
3,0,KeySignature_part_2_1,0
4,0,TimeSignature_part_1_1,0
...,...,...,...
377,0,part_2_voice1_measure76_ev2,0
378,1024,part_1_voice0_measure77_ev0,1024
379,0,part_2_voice0_measure77_ev0,0
380,3072,part_1_voice0_measure78_ev0,3072


## Parsing \<logic\>

In [6]:
staves = dfs["ieee1599.logic.los.staff_list.staff"]
staves.show_meta()
staves

{'ieee1599.@creator': 'Finale Plugin',
 'ieee1599.@version': '1.0',
 'xml_node': 'ieee1599.logic.los.staff_list.staff'}


,@id,@line_number,time_signature.@event_ref,time_signature.time_indication.@den,time_signature.time_indication.@num,key_signature.@event_ref,key_signature.sharp_num.@number,clef.@event_ref,clef.@octave_num,clef.@shape,clef.@staff_step
0,part_1_staff,5,TimeSignature_part_1_1,4,3,KeySignature_part_1_1,2,Clef_part_1_1,0,G,2
1,part_2_staff,5,TimeSignature_part_2_1,4,3,KeySignature_part_2_1,2,Clef_part_2_1,0,F,6


In [7]:
events = dfs["ieee1599.logic.los.part"]
events.show_meta()
events

{'ieee1599.@creator': 'Finale Plugin',
 'ieee1599.@version': '1.0',
 'xml_node': 'ieee1599.logic.los.part'}


,@id,voice_list,measure.@number,measure.voice.@voice_item_ref,item_type,measure.voice.rest.@event_ref,measure.voice.rest.duration.@den,measure.voice.rest.duration.@num,measure.voice.chord.@event_ref,measure.voice.chord.duration.@den,measure.voice.chord.duration.@num,measure.voice.chord.notehead.pitch.@actual_accidental,measure.voice.chord.notehead.pitch.@octave,measure.voice.chord.notehead.pitch.@step,measure.voice.chord.augmentation_dots.@number,measure.voice.chord.notehead.printed_accidentals.natural,measure.voice.chord.notehead
0,part_1,"[{'voice_item': [{'@id': 'part_1_0_voice', '@s...",1,part_1_0_voice,measure.voice.rest,part_1_voice0_measure1_ev0,4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,part_1,"[{'voice_item': [{'@id': 'part_1_0_voice', '@s...",2,part_1_0_voice,measure.voice.rest,part_1_voice0_measure2_ev0,4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,part_1,"[{'voice_item': [{'@id': 'part_1_0_voice', '@s...",3,part_1_0_voice,measure.voice.rest,part_1_voice0_measure3_ev0,4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,part_1,"[{'voice_item': [{'@id': 'part_1_0_voice', '@s...",4,part_1_0_voice,measure.voice.rest,part_1_voice0_measure4_ev0,4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,part_1,"[{'voice_item': [{'@id': 'part_1_0_voice', '@s...",5,part_1_0_voice,measure.voice.rest,part_1_voice0_measure5_ev0,4,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,part_2,"[{'voice_item': [{'@id': 'part_2_0_voice', '@s...",76,part_2_1_voice,measure.voice.rest,part_2_voice1_measure76_ev0,4,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
372,part_2,"[{'voice_item': [{'@id': 'part_2_0_voice', '@s...",76,part_2_1_voice,measure.voice.chord,NaN,NaN,NaN,part_2_voice1_measure76_ev1,4,1,natural,3,B,NaN,NaN,NaN
373,part_2,"[{'voice_item': [{'@id': 'part_2_0_voice', '@s...",76,part_2_1_voice,measure.voice.chord,NaN,NaN,NaN,part_2_voice1_measure76_ev2,4,1,natural,4,E,NaN,NaN,NaN
374,part_2,"[{'voice_item': [{'@id': 'part_2_0_voice', '@s...",77,part_2_0_voice,measure.voice.chord,NaN,NaN,NaN,part_2_voice0_measure77_ev0,2,1,NaN,NaN,NaN,1,NaN,"[{'pitch': {'@actual_accidental': 'natural', '..."


## Parsing \<notational\>

In [8]:
graphic_instances = dfs["ieee1599.notational.graphic_instance_group"]
graphic_instances.show_meta()
graphic_instances

{'ieee1599.@creator': 'Finale Plugin',
 'ieee1599.@version': '1.0',
 'xml_node': 'ieee1599.notational.graphic_instance_group'}


,@description,graphic_instance.@encoding_format,graphic_instance.@file_format,graphic_instance.@file_name,graphic_instance.@measurement_unit,graphic_instance.@position_in_group,graphic_instance.graphic_event.@event_ref,graphic_instance.graphic_event.@lower_right_x,graphic_instance.graphic_event.@lower_right_y,graphic_instance.graphic_event.@upper_left_x,graphic_instance.graphic_event.@upper_left_y
0,eng:Montréal: Les Éditions Outremontaises (2006),image_jpeg,image_jpeg,score/Montreal/IMSLP01599-Satie_Gymnopedies-1.png,pixels,1,Clef_part_1_1,64,164,50,132
1,eng:Montréal: Les Éditions Outremontaises (2006),image_jpeg,image_jpeg,score/Montreal/IMSLP01599-Satie_Gymnopedies-1.png,pixels,1,Clef_part_2_1,64,208,49,192
2,eng:Montréal: Les Éditions Outremontaises (2006),image_jpeg,image_jpeg,score/Montreal/IMSLP01599-Satie_Gymnopedies-1.png,pixels,1,KeySignature_part_1_1,79,153,66,131
3,eng:Montréal: Les Éditions Outremontaises (2006),image_jpeg,image_jpeg,score/Montreal/IMSLP01599-Satie_Gymnopedies-1.png,pixels,1,KeySignature_part_2_1,79,211,66,190
4,eng:Montréal: Les Éditions Outremontaises (2006),image_jpeg,image_jpeg,score/Montreal/IMSLP01599-Satie_Gymnopedies-1.png,pixels,1,TimeSignature_part_1_1,94,158,84,139
...,...,...,...,...,...,...,...,...,...,...,...
759,eng:Transcription (2012),image_jpeg,image_jpeg,score/Transcription/ProgettoLim4.png,pixels,4,part_2_voice1_measure76_ev2,299,618,291,603
760,eng:Transcription (2012),image_jpeg,image_jpeg,score/Transcription/ProgettoLim4.png,pixels,4,part_1_voice0_measure77_ev0,336,582,320,549
761,eng:Transcription (2012),image_jpeg,image_jpeg,score/Transcription/ProgettoLim4.png,pixels,4,part_2_voice0_measure77_ev0,336,635,325,607
762,eng:Transcription (2012),image_jpeg,image_jpeg,score/Transcription/ProgettoLim4.png,pixels,4,part_1_voice0_measure78_ev0,401,576,384,546


## Parsing \<audio\>

In [9]:
audio_times = dfs["ieee1599.audio.track"]
audio_times.show_meta()
audio_times

{'ieee1599.@creator': 'Finale Plugin',
 'ieee1599.@version': '1.0',
 'xml_node': 'ieee1599.audio.track'}


,@encoding_format,@file_format,@file_name,track_general.notes,track_general.performers.performer.@name,track_general.performers.performer.@type,track_indexing.@timing_type,track_indexing.track_event.@event_ref,track_indexing.track_event.@start_time
0,audio_mp3,audio_mp3,audio/ChaseColeman/satie-gymnopedie1-coleman.mp3,Chase Coleman,Chase Coleman,piano,seconds,Clef_part_1_1,2.46
1,audio_mp3,audio_mp3,audio/ChaseColeman/satie-gymnopedie1-coleman.mp3,Chase Coleman,Chase Coleman,piano,seconds,Clef_part_2_1,2.46
2,audio_mp3,audio_mp3,audio/ChaseColeman/satie-gymnopedie1-coleman.mp3,Chase Coleman,Chase Coleman,piano,seconds,KeySignature_part_1_1,2.46
3,audio_mp3,audio_mp3,audio/ChaseColeman/satie-gymnopedie1-coleman.mp3,Chase Coleman,Chase Coleman,piano,seconds,KeySignature_part_2_1,2.46
4,audio_mp3,audio_mp3,audio/ChaseColeman/satie-gymnopedie1-coleman.mp3,Chase Coleman,Chase Coleman,piano,seconds,TimeSignature_part_1_1,2.46
...,...,...,...,...,...,...,...,...,...
759,audio_mp3,audio_mp3,audio/AndreasPfaul/satie-gymnopedie1-pfaul.mp3,Andreas Pfaul,Andreas Pfaul,piano,seconds,part_2_voice1_measure76_ev2,187.08
760,audio_mp3,audio_mp3,audio/AndreasPfaul/satie-gymnopedie1-pfaul.mp3,Andreas Pfaul,Andreas Pfaul,piano,seconds,part_1_voice0_measure77_ev0,188.63
761,audio_mp3,audio_mp3,audio/AndreasPfaul/satie-gymnopedie1-pfaul.mp3,Andreas Pfaul,Andreas Pfaul,piano,seconds,part_2_voice0_measure77_ev0,188.63
762,audio_mp3,audio_mp3,audio/AndreasPfaul/satie-gymnopedie1-pfaul.mp3,Andreas Pfaul,Andreas Pfaul,piano,seconds,part_1_voice0_measure78_ev0,192.22


## Parsing \<structural\>

In [10]:
structural_analysis = dfs['ieee1599.structural.analysis']
structural_analysis.show_meta()
structural_analysis

{'ieee1599.@creator': 'Finale Plugin',
 'ieee1599.@version': '1.0',
 'xml_node': 'ieee1599.structural.analysis'}


,@author,@id,segmentation.segment.@id,segmentation.segment.segment_event.@event_ref,segmentation.segment.segment_event.@event_ref_exploded
0,Simone Delle Fave,Analisi_1,Analisi_1_L1_A,NaN,part_1_voice0_measure1_ev0
1,Simone Delle Fave,Analisi_1,Analisi_1_L1_A,NaN,part_2_voice0_measure1_ev0
2,Simone Delle Fave,Analisi_1,Analisi_1_L1_A,NaN,part_2_voice1_measure1_ev0
3,Simone Delle Fave,Analisi_1,Analisi_1_L1_A,NaN,part_2_voice1_measure1_ev1
4,Simone Delle Fave,Analisi_1,Analisi_1_L1_A,NaN,part_1_voice0_measure2_ev0
...,...,...,...,...,...
1790,Simone Delle Fave,Analisi_2,Analisi_2_L3_RS_F_24,NaN,part_1_voice0_measure73_ev1
1791,Simone Delle Fave,Analisi_2,Analisi_2_L3_RS_F_24,NaN,part_1_voice0_measure73_ev2
1792,Simone Delle Fave,Analisi_2,Analisi_2_L3_RS_F_25,NaN,part_1_voice0_measure74_ev0
1793,Simone Delle Fave,Analisi_2,Analisi_2_L3_RS_F_25,NaN,part_1_voice0_measure74_ev1


In [11]:
petri_nets = dfs['ieee1599.structural.petri_nets']
petri_nets.show_meta()
petri_nets

{'ieee1599.@creator': 'Finale Plugin',
 'ieee1599.@version': '1.0',
 'xml_node': 'ieee1599.structural.petri_nets'}


,petri_net.@file_name,petri_net.place.@place_ref,petri_net.place.@segment_ref,petri_net.place.@place_ref_exploded,petri_net.place.@segment_ref_exploded
0,Analisi_1/L1.pnml,NaN,NaN,p2,Analisi_1_L1_A
1,Analisi_1/L1.pnml,NaN,NaN,p3,Analisi_1_L1_B
2,Analisi_1/L1.pnml,NaN,NaN,p4,Analisi_1_L1_C
3,Analisi_1/L1.pnml,NaN,NaN,p5,Analisi_1_L1_D
4,Analisi_1/L1.pnml,NaN,NaN,p6,Analisi_1_L1_E
...,...,...,...,...,...
92,Analisi_2/L3-RS-D.pnml,NaN,NaN,p4,Analisi_2_L3_RS_D_19
93,Analisi_2/L3-RS-F.pnml,NaN,NaN,p7,Analisi_2_L3_RS_F_17
94,Analisi_2/L3-RS-F.pnml,NaN,NaN,p2,Analisi_2_L3_RS_F_23
95,Analisi_2/L3-RS-F.pnml,NaN,NaN,p3,Analisi_2_L3_RS_F_24


## Assembling

The idea here is to create one big DataFrame that has the spine's event IDs as index and joins the coordinates/features from the various domains one after the other.
The procedure amounts to the same for each domain but requires different transformations each time.
Roughly speaking:

1. the columns holding the relevant information need to be selected;
2. when the IEEE1599 file contains more than one file for a domain, a grouper needs to be selected/constructed;
3. one DataFrame per file with event-ID index is created, typically in a way that the column names specify which file they concern;
4. each DataFrame is joined to the `alignment` DataFrame which will hold the end result.

### Spine

In [12]:
alignment = (
    spine.set_index("@id")
    .astype("Int64")
    .rename_axis("event_id")
    .set_axis(["timing", "hpos"], axis=1)
)
alignment = pd.concat(
    [alignment, (alignment.cumsum().set_axis(["timing_cum", "hpos_cum"], axis=1))],
    axis=1,
)
print(f"The spine consists of {len(alignment)} events.")
alignment

The spine consists of 382 events.


,timing,hpos,timing_cum,hpos_cum
event_id,,,,
Clef_part_1_1,0,0,0,0
Clef_part_2_1,0,0,0,0
KeySignature_part_1_1,0,0,0,0
KeySignature_part_2_1,0,0,0,0
TimeSignature_part_1_1,0,0,0,0
...,...,...,...,...
part_2_voice1_measure76_ev2,0,0,232448,232448
part_1_voice0_measure77_ev0,1024,1024,233472,233472
part_2_voice0_measure77_ev0,0,0,233472,233472


### Pitches and rests

In [13]:
events.columns

Index(['@id', 'voice_list', 'measure.@number', 'measure.voice.@voice_item_ref',
       'item_type', 'measure.voice.rest.@event_ref',
       'measure.voice.rest.duration.@den', 'measure.voice.rest.duration.@num',
       'measure.voice.chord.@event_ref', 'measure.voice.chord.duration.@den',
       'measure.voice.chord.duration.@num',
       'measure.voice.chord.notehead.pitch.@actual_accidental',
       'measure.voice.chord.notehead.pitch.@octave',
       'measure.voice.chord.notehead.pitch.@step',
       'measure.voice.chord.augmentation_dots.@number',
       'measure.voice.chord.notehead.printed_accidentals.natural',
       'measure.voice.chord.notehead'],
      dtype='object')

In [14]:
def unite_chord_and_rest_columns(
    events: pd.DataFrame, chord_column: str, new_name: str
):
    rest_column = chord_column.replace("chord", "rest")
    return events[chord_column].fillna(events[rest_column]).rename(new_name)


def make_duration_column(events, chord_num_col, chord_den_col):
    dur_num = unite_chord_and_rest_columns(events, chord_num_col, "dur_num")
    dur_den = unite_chord_and_rest_columns(events, chord_den_col, "dur_den")
    return dur_num + "/" + dur_den


def make_pitch_column(events, step_col, octave_col, accidental_col):
    accidental = (
        events[accidental_col].map(dict(natural="", flat="b", sharp="#")).fillna("")
    )
    return (events[step_col] + accidental + events[octave_col]).fillna("rest")

In [15]:
chord_num_col = "measure.voice.chord.duration.@num"
chord_den_col = "measure.voice.chord.duration.@den"
duration = make_duration_column(events, chord_num_col, chord_den_col)

step_col = "measure.voice.chord.notehead.pitch.@step"
accidental_col = "measure.voice.chord.notehead.pitch.@actual_accidental"
octave_col = "measure.voice.chord.notehead.pitch.@octave"
pitch = make_pitch_column(events, step_col, octave_col, accidental_col)

pitches_and_rests = pd.DataFrame(
    dict(
        event_id=unite_chord_and_rest_columns(
            events, "measure.voice.chord.@event_ref", "event_id"
        ),
        pitch=pitch,
        duration=duration,
    )
).set_index("event_id")
pitches_and_rests

,pitch,duration
event_id,,
part_1_voice0_measure1_ev0,rest,3/4
part_1_voice0_measure2_ev0,rest,3/4
part_1_voice0_measure3_ev0,rest,3/4
part_1_voice0_measure4_ev0,rest,3/4
part_1_voice0_measure5_ev0,rest,1/4
...,...,...
part_2_voice1_measure76_ev0,rest,1/4
part_2_voice1_measure76_ev1,B3,1/4
part_2_voice1_measure76_ev2,E4,1/4


In [16]:
def join_and_test(alignment: pd.DataFrame, right_side: pd.DataFrame, name: str):
    result = alignment.join(right_side, how="outer").sort_values(
        ["timing_cum", "pitch"]
    )
    assert (
        result.timing.notna().all()
    ), f"{name} contains events not included in the spine"
    return result


if "pitch" not in alignment:
    alignment = join_and_test(alignment, pitches_and_rests, "LOS")
alignment

,timing,hpos,timing_cum,hpos_cum,pitch,duration
event_id,,,,,,
part_2_voice0_measure1_ev0,0,0,0,0,G3,1/2
part_1_voice0_measure1_ev0,0,0,0,0,rest,3/4
part_2_voice1_measure1_ev0,0,0,0,0,rest,1/4
Clef_part_1_1,0,0,0,0,NaN,NaN
Clef_part_2_1,0,0,0,0,NaN,NaN
...,...,...,...,...,...,...
part_1_voice1_measure76_ev2,1024,1024,232448,232448,rest,1/4
part_1_voice0_measure77_ev0,1024,1024,233472,233472,rest,1/2
part_2_voice0_measure77_ev0,0,0,233472,233472,rest,1/2


### Notational

In [17]:
graphic_instances.columns

Index(['@description', 'graphic_instance.@encoding_format',
       'graphic_instance.@file_format', 'graphic_instance.@file_name',
       'graphic_instance.@measurement_unit',
       'graphic_instance.@position_in_group',
       'graphic_instance.graphic_event.@event_ref',
       'graphic_instance.graphic_event.@lower_right_x',
       'graphic_instance.graphic_event.@lower_right_y',
       'graphic_instance.graphic_event.@upper_left_x',
       'graphic_instance.graphic_event.@upper_left_y'],
      dtype='object')

In [18]:
filename_col = "graphic_instance.@file_name"

filename = graphic_instances[filename_col].map(lambda path: path.split("/")[-1])
filename.value_counts()

IMSLP01599-Satie_Gymnopedies-2.png    108
ProgettoLim2.png                      108
ProgettoLim3.png                       96
IMSLP01599-Satie_Gymnopedies-3.png     96
ProgettoLim4.png                       92
IMSLP01599-Satie_Gymnopedies-4.png     92
IMSLP01599-Satie_Gymnopedies-1.png     86
ProgettoLim1.png                       86
Name: count, dtype: int64

In [19]:
event_ref_col = "graphic_instance.graphic_event.@event_ref"

coordinates = pd.DataFrame(
    dict(
        event_id=graphic_instances[event_ref_col],
        filename=filename,
        coordinates=[
            dict(upper_left=dict(x=ulx, y=uly), lower_right=dict(x=lrx, y=lry))
            for ulx, uly, lrx, lry in graphic_instances.iloc[:, -4:]
            .astype("Int64")
            .itertuples(index=False)
        ],
    )
)
coordinates

,event_id,filename,coordinates
0,Clef_part_1_1,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 64, 'y': 164}, 'lower_rig..."
1,Clef_part_2_1,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 64, 'y': 208}, 'lower_rig..."
2,KeySignature_part_1_1,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 79, 'y': 153}, 'lower_rig..."
3,KeySignature_part_2_1,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 79, 'y': 211}, 'lower_rig..."
4,TimeSignature_part_1_1,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 94, 'y': 158}, 'lower_rig..."
...,...,...,...
759,part_2_voice1_measure76_ev2,ProgettoLim4.png,"{'upper_left': {'x': 299, 'y': 618}, 'lower_ri..."
760,part_1_voice0_measure77_ev0,ProgettoLim4.png,"{'upper_left': {'x': 336, 'y': 582}, 'lower_ri..."
761,part_2_voice0_measure77_ev0,ProgettoLim4.png,"{'upper_left': {'x': 336, 'y': 635}, 'lower_ri..."
762,part_1_voice0_measure78_ev0,ProgettoLim4.png,"{'upper_left': {'x': 401, 'y': 576}, 'lower_ri..."


In [20]:
def group_coordinates_by_editions(
    coordinates: pd.DataFrame, edition_grouper: pd.Series
) -> dict[str, pd.DataFrame]:
    edition_coordinates = {}
    for edition, ed_coords in coordinates.groupby(edition_grouper):
        ed_coords = ed_coords.set_index("event_id").add_prefix(f"{edition}_")
        edition_coordinates[edition] = ed_coords
    return edition_coordinates


edition_grouper = filename.str[:-6]
# the 'ieee1599.notational.graphic_instance_group.@description' column fulfils the
# role of grouping editions but the names would not make for good column titles
edition_coordinates = group_coordinates_by_editions(coordinates, edition_grouper)
if not alignment.columns.str.endswith("coordinates").any():
    for edition, ed_coords in edition_coordinates.items():
        alignment = join_and_test(alignment, ed_coords, edition)
alignment

,timing,hpos,timing_cum,hpos_cum,pitch,duration,IMSLP01599-Satie_Gymnopedies_filename,IMSLP01599-Satie_Gymnopedies_coordinates,ProgettoLi_filename,ProgettoLi_coordinates
event_id,,,,,,,,,,
part_2_voice0_measure1_ev0,0,0,0,0,G3,1/2,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 115, 'y': 225}, 'lower_ri...",ProgettoLim1.png,"{'upper_left': {'x': 131, 'y': 197}, 'lower_ri..."
part_1_voice0_measure1_ev0,0,0,0,0,rest,3/4,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 145, 'y': 147}, 'lower_ri...",ProgettoLim1.png,"{'upper_left': {'x': 160, 'y': 123}, 'lower_ri..."
part_2_voice1_measure1_ev0,0,0,0,0,rest,1/4,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 111, 'y': 200}, 'lower_ri...",ProgettoLim1.png,"{'upper_left': {'x': 127, 'y': 175}, 'lower_ri..."
Clef_part_1_1,0,0,0,0,NaN,NaN,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 64, 'y': 164}, 'lower_rig...",ProgettoLim1.png,"{'upper_left': {'x': 89, 'y': 140}, 'lower_rig..."
Clef_part_2_1,0,0,0,0,NaN,NaN,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 64, 'y': 208}, 'lower_rig...",ProgettoLim1.png,"{'upper_left': {'x': 89, 'y': 184}, 'lower_rig..."
...,...,...,...,...,...,...,...,...,...,...
part_1_voice1_measure76_ev2,1024,1024,232448,232448,rest,1/4,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 305, 'y': 550}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 300, 'y': 588}, 'lower_ri..."
part_1_voice0_measure77_ev0,1024,1024,233472,233472,rest,1/2,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 350, 'y': 541}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 336, 'y': 582}, 'lower_ri..."
part_2_voice0_measure77_ev0,0,0,233472,233472,rest,1/2,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 349, 'y': 601}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 336, 'y': 635}, 'lower_ri..."


### Audio

In [21]:
audio_times.columns

Index(['@encoding_format', '@file_format', '@file_name', 'track_general.notes',
       'track_general.performers.performer.@name',
       'track_general.performers.performer.@type',
       'track_indexing.@timing_type', 'track_indexing.track_event.@event_ref',
       'track_indexing.track_event.@start_time'],
      dtype='object')

In [22]:
audio_event_ref_col = "track_indexing.track_event.@event_ref"
audio_filename_col = "@file_name"
start_time_col = "track_indexing.track_event.@start_time"
timing_type_col = "track_indexing.@timing_type"
audio_times[audio_filename_col].value_counts()

@file_name
audio/ChaseColeman/satie-gymnopedie1-coleman.mp3    382
audio/AndreasPfaul/satie-gymnopedie1-pfaul.mp3      382
Name: count, dtype: int64

In [23]:
audio_names = (
    audio_times[audio_filename_col].str.slice(stop=-4).str.split("/", expand=True)[1]
)

start_times = pd.DataFrame(
    dict(
        event_id=audio_times[audio_event_ref_col],
        filename=audio_names,
        start_time=audio_times[start_time_col],
    )
).set_index("event_id")
start_times

,filename,start_time
event_id,,
Clef_part_1_1,ChaseColeman,2.46
Clef_part_2_1,ChaseColeman,2.46
KeySignature_part_1_1,ChaseColeman,2.46
KeySignature_part_2_1,ChaseColeman,2.46
TimeSignature_part_1_1,ChaseColeman,2.46
...,...,...
part_2_voice1_measure76_ev2,AndreasPfaul,187.08
part_1_voice0_measure77_ev0,AndreasPfaul,188.63
part_2_voice0_measure77_ev0,AndreasPfaul,188.63


In [24]:
def group_start_times_by_tracks(
    start_times: pd.DataFrame, track_grouper: str
) -> dict[str, pd.DataFrame]:
    """The difference: group_coordinates_by_editions() uses a grouper series external
    to the grouped DataFrame, hence the index needs to be unique and is only set to "event_id"
    for each group dataframe individually. Here, the grouper is one of the columns,
    so this extra step is not needed and the index can be set before grouping. In
    a generalized converter, the functionality should be merged, and combined with an automated
    grouping based on the respective @description fields.
    """
    track_times = {}
    for track, times in start_times.groupby(track_grouper):
        times = times.add_prefix(f"{track}_")
        track_times[track] = times
    return track_times


track_times = group_start_times_by_tracks(start_times, "filename")
if not alignment.columns.str.endswith("start_time").any():
    for track, times in track_times.items():
        alignment = join_and_test(alignment, times, track)
alignment

,timing,hpos,timing_cum,hpos_cum,pitch,duration,IMSLP01599-Satie_Gymnopedies_filename,IMSLP01599-Satie_Gymnopedies_coordinates,ProgettoLi_filename,ProgettoLi_coordinates,AndreasPfaul_filename,AndreasPfaul_start_time,ChaseColeman_filename,ChaseColeman_start_time
event_id,,,,,,,,,,,,,,
part_2_voice0_measure1_ev0,0,0,0,0,G3,1/2,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 115, 'y': 225}, 'lower_ri...",ProgettoLim1.png,"{'upper_left': {'x': 131, 'y': 197}, 'lower_ri...",AndreasPfaul,0.5,ChaseColeman,2.46
part_1_voice0_measure1_ev0,0,0,0,0,rest,3/4,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 145, 'y': 147}, 'lower_ri...",ProgettoLim1.png,"{'upper_left': {'x': 160, 'y': 123}, 'lower_ri...",AndreasPfaul,0.5,ChaseColeman,2.46
part_2_voice1_measure1_ev0,0,0,0,0,rest,1/4,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 111, 'y': 200}, 'lower_ri...",ProgettoLim1.png,"{'upper_left': {'x': 127, 'y': 175}, 'lower_ri...",AndreasPfaul,0.5,ChaseColeman,2.46
Clef_part_1_1,0,0,0,0,NaN,NaN,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 64, 'y': 164}, 'lower_rig...",ProgettoLim1.png,"{'upper_left': {'x': 89, 'y': 140}, 'lower_rig...",AndreasPfaul,0.5,ChaseColeman,2.46
Clef_part_2_1,0,0,0,0,NaN,NaN,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 64, 'y': 208}, 'lower_rig...",ProgettoLim1.png,"{'upper_left': {'x': 89, 'y': 184}, 'lower_rig...",AndreasPfaul,0.5,ChaseColeman,2.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
part_1_voice1_measure76_ev2,1024,1024,232448,232448,rest,1/4,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 305, 'y': 550}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 300, 'y': 588}, 'lower_ri...",AndreasPfaul,187.08,ChaseColeman,184.21
part_1_voice0_measure77_ev0,1024,1024,233472,233472,rest,1/2,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 350, 'y': 541}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 336, 'y': 582}, 'lower_ri...",AndreasPfaul,188.63,ChaseColeman,185.27
part_2_voice0_measure77_ev0,0,0,233472,233472,rest,1/2,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 349, 'y': 601}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 336, 'y': 635}, 'lower_ri...",AndreasPfaul,188.63,ChaseColeman,185.27


### Structural

The analysis lists "segments" which correspond to Petri net nodes, and lists for each of them the spine event IDs (`<segment_event>`) that occur under that node.
In order to include this information in the `alignment` DataFrame, we group this data by analyses (there are 2) and by event ID in order to attribute to each of them
all segments/nodes under which it occurs. In other words, for each Petri-net analysis, we attribute to each event a list of segment IDs.

In addition, the IEEE1599 file links each segment to a node (`<place>`) in one of the graphical `.pnml` Petri-net encodings.
The parsed information is contained in the `petri_net` DataFrame, but since it is a 1:1 mapping, we refrain from including it here.

In [25]:
structural_analysis.columns

Index(['@author', '@id', 'segmentation.segment.@id',
       'segmentation.segment.segment_event.@event_ref',
       'segmentation.segment.segment_event.@event_ref_exploded'],
      dtype='object')

In [26]:
analysis_id_col = "@id"
analysis_event_ref_col = "segmentation.segment.segment_event.@event_ref_exploded"
analysis_single_event_ref_col = "segmentation.segment.segment_event.@event_ref"
segment_id_col = "segmentation.segment.@id"

analysis_segments = pd.DataFrame(dict(
    event_ref = structural_analysis[analysis_event_ref_col].fillna(structural_analysis[analysis_single_event_ref_col]),
    analysis_id = structural_analysis[analysis_id_col],
    segment_id = structural_analysis[segment_id_col],
))
analysis_segments

,event_ref,analysis_id,segment_id
0,part_1_voice0_measure1_ev0,Analisi_1,Analisi_1_L1_A
1,part_2_voice0_measure1_ev0,Analisi_1,Analisi_1_L1_A
2,part_2_voice1_measure1_ev0,Analisi_1,Analisi_1_L1_A
3,part_2_voice1_measure1_ev1,Analisi_1,Analisi_1_L1_A
4,part_1_voice0_measure2_ev0,Analisi_1,Analisi_1_L1_A
...,...,...,...
1790,part_1_voice0_measure73_ev1,Analisi_2,Analisi_2_L3_RS_F_24
1791,part_1_voice0_measure73_ev2,Analisi_2,Analisi_2_L3_RS_F_24
1792,part_1_voice0_measure74_ev0,Analisi_2,Analisi_2_L3_RS_F_25
1793,part_1_voice0_measure74_ev1,Analisi_2,Analisi_2_L3_RS_F_25


In [27]:
def group_event_refs_by_analysis_segment(
    analysis_segments: pd.DataFrame,
    analysis_grouper: str = "analysis_id"
) -> dict[str, pd.DataFrame]:
    event_ref2segments_dfs = {}
    for analysis, segments in analysis_segments.groupby(analysis_grouper):
        events2segments = segments.groupby("event_ref").segment_id.apply(list)
        events2segments = events2segments.rename(f"{analysis}_segment_ids")
        event_ref2segments_dfs[analysis] = events2segments
    return event_ref2segments_dfs


event_ref2segments_dfs = group_event_refs_by_analysis_segment(analysis_segments)
if not alignment.columns.str.endswith("segment_ids").any():
    for analysis, segments in event_ref2segments_dfs.items():
        alignment = join_and_test(alignment, segments, analysis)
alignment

,timing,hpos,timing_cum,hpos_cum,pitch,duration,IMSLP01599-Satie_Gymnopedies_filename,IMSLP01599-Satie_Gymnopedies_coordinates,ProgettoLi_filename,ProgettoLi_coordinates,AndreasPfaul_filename,AndreasPfaul_start_time,ChaseColeman_filename,ChaseColeman_start_time,Analisi_1_segment_ids,Analisi_2_segment_ids
part_2_voice0_measure1_ev0,0,0,0,0,G3,1/2,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 115, 'y': 225}, 'lower_ri...",ProgettoLim1.png,"{'upper_left': {'x': 131, 'y': 197}, 'lower_ri...",AndreasPfaul,0.5,ChaseColeman,2.46,"[Analisi_1_L1_A, Analisi_1_L2_A_RI, Analisi_1_...","[Analisi_2_L1_RI, Analisi_2_L2_RI_A, Analisi_2..."
part_1_voice0_measure1_ev0,0,0,0,0,rest,3/4,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 145, 'y': 147}, 'lower_ri...",ProgettoLim1.png,"{'upper_left': {'x': 160, 'y': 123}, 'lower_ri...",AndreasPfaul,0.5,ChaseColeman,2.46,"[Analisi_1_L1_A, Analisi_1_L2_A_RS, Analisi_1_...","[Analisi_2_L1_RS, Analisi_2_L2_RS_A, Analisi_2..."
part_2_voice1_measure1_ev0,0,0,0,0,rest,1/4,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 111, 'y': 200}, 'lower_ri...",ProgettoLim1.png,"{'upper_left': {'x': 127, 'y': 175}, 'lower_ri...",AndreasPfaul,0.5,ChaseColeman,2.46,"[Analisi_1_L1_A, Analisi_1_L2_A_RI, Analisi_1_...","[Analisi_2_L1_RI, Analisi_2_L2_RI_A, Analisi_2..."
Clef_part_1_1,0,0,0,0,NaN,NaN,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 64, 'y': 164}, 'lower_rig...",ProgettoLim1.png,"{'upper_left': {'x': 89, 'y': 140}, 'lower_rig...",AndreasPfaul,0.5,ChaseColeman,2.46,NaN,NaN
Clef_part_2_1,0,0,0,0,NaN,NaN,IMSLP01599-Satie_Gymnopedies-1.png,"{'upper_left': {'x': 64, 'y': 208}, 'lower_rig...",ProgettoLim1.png,"{'upper_left': {'x': 89, 'y': 184}, 'lower_rig...",AndreasPfaul,0.5,ChaseColeman,2.46,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
part_1_voice1_measure76_ev2,1024,1024,232448,232448,rest,1/4,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 305, 'y': 550}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 300, 'y': 588}, 'lower_ri...",AndreasPfaul,187.08,ChaseColeman,184.21,"[Analisi_1_L1_L, Analisi_1_L2_L_RS]","[Analisi_2_L1_RS, Analisi_2_L2_RS_G]"
part_1_voice0_measure77_ev0,1024,1024,233472,233472,rest,1/2,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 350, 'y': 541}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 336, 'y': 582}, 'lower_ri...",AndreasPfaul,188.63,ChaseColeman,185.27,"[Analisi_1_L1_L, Analisi_1_L2_L_RS]","[Analisi_2_L1_RS, Analisi_2_L2_RS_G]"
part_2_voice0_measure77_ev0,0,0,233472,233472,rest,1/2,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 349, 'y': 601}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 336, 'y': 635}, 'lower_ri...",AndreasPfaul,188.63,ChaseColeman,185.27,"[Analisi_1_L1_L, Analisi_1_L2_L_RI]","[Analisi_2_L1_RI, Analisi_2_L2_RI_E]"
part_1_voice0_measure78_ev0,3072,3072,236544,236544,rest,1/2,IMSLP01599-Satie_Gymnopedies-4.png,"{'upper_left': {'x': 424, 'y': 535}, 'lower_ri...",ProgettoLim4.png,"{'upper_left': {'x': 401, 'y': 576}, 'lower_ri...",AndreasPfaul,192.22,ChaseColeman,188.77,"[Analisi_1_L1_L, Analisi_1_L2_L_RS]","[Analisi_2_L1_RS, Analisi_2_L2_RS_G]"


## Storing to Parquet

In [28]:
table = pa.Table.from_pandas(alignment)
pretty_print_table_schema(table)

{'Analisi_1_segment_ids': ['string'],
 'Analisi_2_segment_ids': ['string'],
 'AndreasPfaul_filename': 'string',
 'AndreasPfaul_start_time': 'string',
 'ChaseColeman_filename': 'string',
 'ChaseColeman_start_time': 'string',
 'IMSLP01599-Satie_Gymnopedies_coordinates': {'lower_right': {'x': 'int64',
                                                              'y': 'int64'},
                                              'upper_left': {'x': 'int64',
                                                             'y': 'int64'}},
 'IMSLP01599-Satie_Gymnopedies_filename': 'string',
 'ProgettoLi_coordinates': {'lower_right': {'x': 'int64', 'y': 'int64'},
                            'upper_left': {'x': 'int64', 'y': 'int64'}},
 'ProgettoLi_filename': 'string',
 '__index_level_0__': 'string',
 'duration': 'string',
 'hpos': 'int64',
 'hpos_cum': 'int64',
 'pitch': 'string',
 'timing': 'int64',
 'timing_cum': 'int64'}


In [29]:
def add_metadata(
    table: pa.Table,
    metadata: Optional[dict[str, str]] = None,
    col2metadata: Optional[dict[str, dict[str, str]]] = None,
) -> pa.Table:
    """Returns a copy of the table with metadata added to its overall schema or
    to the fields representing the columns specified as keys in `col2metadata`.
    """
    if metadata is None and col2metadata is None:
        return table
    schema = table.schema
    if metadata:
        schema = schema.with_metadata(metadata)
    if col2metadata:
        for column, md in col2metadata.items():
            field_index = schema.get_field_index(column)
            new_field = schema[field_index].with_metadata(md)
            schema = schema.set(field_index, new_field)
    return table.cast(schema)


field_metadata = {name: dict(measurement_unit="pixels") for name in table.schema.names}
table = add_metadata(table, col2metadata=field_metadata)

**Parquet roundtrip**

In [30]:
parquet_filepath = utils.make_output_path("gymnopedie_01.parquet")
pq.write_table(
    table, parquet_filepath, compression="snappy", version="2.6", row_group_size=100000
)

In [31]:
table = pq.read_table(parquet_filepath)

**Accessing a coordinate column's metadata**

In [32]:
table.field("ProgettoLi_coordinates").metadata

{b'measurement_unit': b'pixels'}

**Retrieving the left upper x-coordinate from the StructArray (column of nested dictionaries)**

In [33]:
table["ProgettoLi_coordinates"].type

StructType(struct<upper_left: struct<x: int64, y: int64>, lower_right: struct<x: int64, y: int64>>)

In [34]:
pc.struct_field(table["ProgettoLi_coordinates"], ["upper_left", "x"])

[
  [
    131,
    160,
    127,
    89,
    89,
    ...
    300,
    336,
    336,
    401,
    402
  ]
]